# ASTERIX `.ast` notebook: decoding `.ast` to `.csv`

This notebook is a **structured learning notebook** for ASTERIX binary files.  
It is **not** a full decoder yet. Its goal is to explain the **core message structure** that must be understood before converting `.ast` files into clean tabular data.

## 1. Outer structure of an ASTERIX file

An ASTERIX `.ast` file is usually a **continuous binary stream** made of consecutive ASTERIX message blocks.  
There is no explicit end marker. Each block defines its own size through the `LEN` field.

Each message block begins with:

- **Octet 1** → `CAT` (category)
- **Octets 2–3** → `LEN` (total block length, including `CAT` and `LEN`)
- **Remaining octets** → **record area**

### Message schema

```text
| [CAT][LEN][record area] | [CAT][LEN][record area] | [CAT][LEN][record area] | ...
```

```text
Message block
├── CAT
├── LEN
└── Record area
```

The parser therefore advances block by block using:

```text
read CAT -> read LEN -> extract LEN bytes -> move to next block
```

---

## 2. Internal structure of the record area

The record area contains **one or more records**.  
Each record starts with an **FSPEC** and is followed by the **Data Fields** announced by that FSPEC.

### Records schema

```text
Record
├── FSPEC
├── Data Field
├── Data Field
└── ...
```

```text
ASTRIX block
├── CAT
├── LEN
└── Record area
    ├── Record 1
    │   ├── FSPEC
    │   ├── Data Field
    │   └── ...
    ├── Record 2
    │   ├── FSPEC
    │   ├── Data Field
    │   └── ...
    └── ...
```

Key idea:  
**The FSPEC comes first, and the Data Fields follow in the order defined by the category profile.**

---

## 3. What the FSPEC does

The **FSPEC** (*Field Specification*) is a **presence map**.  
It does not contain the actual values. Its job is to say **which fields are present** in the current record.

Each FSPEC octet contains:

- **7 field presence bits**
- **1 FX bit**

If `FX = 1`, another FSPEC octet follows.  
If `FX = 0`, the FSPEC ends.

### FSPEC schema

```text
| FRN1 | FRN2 | FRN3 | FRN4 | FRN5 | FRN6 | FRN7 | FX | FRN8 | FRN9 | ...
```

```text
FSPEC parsing
├── Read first FSPEC octet
├── Check FX
├── If FX = 1 -> read next FSPEC octet
└── Continue until FX = 0
```

The FSPEC tells us which FRNs are present. FRNs are field positions, not yet semantic field names.
After the FSPEC, the actual Data Fields are transmitted in **increasing FRN order**.

### Example conceptually: 
then the record body is organized as:
```text
          | FRN1 | FRN2 | FRN3 | FRN4 | FRN5 | FRN6 | FRN7 |  FX  |
(FSPEC)   | [1]  | [0]  | [1]  | [0]  | [0]  | [1]  | [0]  |  [0] |

Then based on that FSPEC the fields that the remaining octets of that record will be:
 Data           
Fields    | Field for FRN 1 | Field for FRN 3 | Field for FRN 6 |
```

---

## 4. Data Field to Data Item using UAP tables

These two concepts are different:

- **Data Field** → binary representation transmitted in the record. Ex: PRN1, PRN4, PRN9... 
- **Data Item** → logical information defined by the specification. Ex: I021/010 (Data Source Identification), I021/040 (Time of Day)...

The **UAP** (*User Application Profile*) is the category-specific table that maps Data Fields to Data Items (each CATegori has its own mapping):

### Example conceptually: 

```text
    UAP table
FRN 1 ----> I021/010
FRN 2 ----> I021/040
FRN 3 ----> I021/161
```
---

## 5. Data Items length

Once a Data Item is identified, the decoder must know **how many bytes belong to its Data Field**.

This length can be classified in 5 different types and this is specified in the EUROCONTROL documentation:

### Fixed length
Always the same size.

```text
[2 octets always]
```

### Extended
A primary part may continue into one or more extension parts using `FX`.

```text
[Primary][FX=1][Extension][FX=0]
```

### Explicit length
The field begins with a length indicator.

```text
[LENGTH][data...]
```

### Repetitive
The field starts with `REP`, followed by repeated substructures.

```text
[REP][element][element][element]
```

### Compound
The field contains internal subfields controlled by an internal presence map.

```text
[Primary subfield][Subfield 1][Subfield 3][Subfield 5]
```

### Clarifying schema

```text
Data Field formats
├── Fixed length
├── Extended
├── Explicit length
├── Repetitive
└── Compound
```

#### **IMPORTANT WARNING**

A single ASTERIX message may contain *more than one record*.  
However, after the FSPEC, the record body may contain variable-length fields.
So the decoder cannot safely jump to *Record 2* unless it knows the exact total size of *Record 1*.

---

## 6. Decoding Data Items

Once each data item length is determined, they must be separated in order to extract the information they are transmiting.
To do so, the EUROCONTROL documentations provide how each item must be analyzed in order to decode the information properly.
Each item has its own format so be careful understanding the structure of each item.

### Example conceptually: 
This example shows how a single octet can be split into several subfields.  
Each bit or group of bits has a defined meaning.

```text
 8   7   6   5   4   3   2   1
+---+---+---+---+---+---+---+---+
|    ATP    | ARC | RC|RAB| FX |
+-----------+-----+---+---+----+
   bits 8/6   5/4   3   2    1

ATP = Reserved for future use
    0 -> 24-Bit ICAO address
    1 -> Duplicate address
    2 -> Surface vehicle address
    3 -> Anonymous address
    4-7 -> Reserved for future use
ARC = Unknown
    0 -> 25 ft
    1 -> 100 ft
    2 -> Unknown
    3 -> Invalid
RC  = Range Check passed, CPR Validation pending
    0 -> Default
    1 -> Range Check passed, CPR Validation pending
RAB = Report from target transponder
    0 -> Report from target transponder
    1 -> Report from field monitor (fixed transponder)
FX  = Extension into first extension
    0 -> End of item
    1 -> Extension into first extension
```

---
## FINAL CLARIFYING SCHEMA

```text
ASTRIX `.ast` file
├── Message block
│   ├── CAT (cat021 or cat cat048)
│   ├── LEN
│   └── Record area
│       ├── Record
│       │   ├── FSPEC --> FRNs detected
│       │   │                UAP tables lookup                
│       │   ├── Data Fields (FRNx) -->  Data Item (I021/020 - Emitter Category)
│       │   ├── Data Fields (FRNy) -->  Data Item (I021/145 - Flight Level)
│       │   └── Data Fields (FRNz) -->  Data Item (I021/230 - Roll Angle)
│       └── ...
└── Next message block
```

This is the correct foundation before decoding values and exporting them into `.csv`.

---
---

## Block 1 — Load the input ASTERIX binary file
This cell imports the required modules, finds the ASTERIX binary input file, reads it into memory, performs a basic size validation, and prints a quick summary of the loaded file and its first header.

In [1]:
import inspect
from pathlib import Path
 
import pandas as pd
 
from asterix_decoder.data_items.data_item import ItemXXX
 
 
# ── Helpers ───────────────────────────────────────────────────────────────────
 
def first_existing_path(candidates: list) -> Path | None:
    """Return the first path in the list that exists on disk."""
    for path in map(Path, candidates):
        if path.exists():
            return path
    return None
 
 
 
BINARY_PATH = first_existing_path([
    Path("raw_data/datos_asterix_adsb.ast"),
])
if BINARY_PATH is None:
    raise FileNotFoundError(
        "No ASTERIX binary file found. Update the path candidates."
    )
 
raw_data: bytes = BINARY_PATH.read_bytes()
 
if len(raw_data) < 3:
    raise ValueError("File is too short to contain a valid ASTERIX header.")
 
print(f"Loaded : {BINARY_PATH}")
print(f"Size   : {len(raw_data):,} bytes")
print(f"First header → CAT={raw_data[0]} | LEN={int.from_bytes(raw_data[1:3], 'big')}")

Loaded : raw_data/datos_asterix_adsb.ast
Size   : 86,455,774 bytes
First header → CAT=21 | LEN=90


## Block 2 — Split the binary stream into ASTERIX messages
This cell defines a parser that walks through the raw binary stream, extracts each message using its header length, stores the message metadata and payload in a DataFrame, and prints the number of messages found per category.

In [2]:
def split_messages(raw: bytes) -> pd.DataFrame:
    """Walk the binary stream and return a DataFrame with one row per message."""
    view = memoryview(raw)
    total = len(raw)
    rows, offset, msg_id = [], 0, 1
 
    while offset + 3 <= total:
        cat    = view[offset]
        length = int.from_bytes(view[offset + 1:offset + 3], "big")
 
        if length < 3 or offset + length > total:
            raise ValueError(f"Invalid message at offset {offset}: LEN={length}")
 
        rows.append({
            "message_id":  msg_id,
            "offset":      offset,
            "cat":         cat,
            "length":      length,
            "data_record": bytes(view[offset + 3:offset + length]),
        })
        offset += length
        msg_id += 1
 
    return pd.DataFrame(rows)
 
 
messages_df = split_messages(raw_data)
 
print(f"\nMessages found: {len(messages_df):,}")
print(messages_df["cat"].value_counts().sort_index().rename("count").rename_axis("CAT").to_string())


Messages found: 975,101
CAT
21    975101


## Block 3 — Parse the FSPEC of each message
This cell precomputes lookup tables for FSPEC decoding, parses the active FRNs and remaining data fields for each message, appends that information to the main DataFrame, and shows a preview of the parsed structure.

In [3]:
_ACTIVE_OFFSETS: tuple[tuple[int, ...], ...] = tuple(
    tuple(i for i in range(7) if (b >> (7 - i)) & 1)
    for b in range(256)
)
# Precompute FX bit too (least-significant bit signals continuation)
_HAS_FX: tuple[bool, ...] = tuple(bool(b & 1) for b in range(256))


def parse_fspec(data_record: bytes) -> tuple[list[int], bytes]:
    frns     : list[int] = []
    frn_base : int       = 1
    cursor   : int       = 0
    n        : int       = len(data_record)

    while cursor < n:
        b = data_record[cursor]
        offsets = _ACTIVE_OFFSETS[b]          # O(1) lookup, no loop
        if offsets:
            base = frn_base                   # avoid repeated attr lookup
            frns += [base + off for off in offsets]
        cursor   += 1
        frn_base += 7
        if not _HAS_FX[b]:                    # O(1) lookup
            break

    return frns, data_record[cursor:]


records = messages_df["data_record"].tolist()        # one allocation, then plain list
parsed  = list(map(parse_fspec, records))            # pure Python map, no pandas tax

messages_df = messages_df.assign(
    frns        = [p[0] for p in parsed],
    data_fields = [p[1] for p in parsed],
)

print(messages_df[["message_id", "cat", "frns", "data_fields"]].head(8).to_string(index=False))

 

 message_id  cat                                                                                                frns                                                                                                                                                                                                                                                                                     data_fields
          1   21     [1, 2, 3, 4, 7, 11, 12, 14, 16, 17, 18, 21, 23, 24, 26, 28, 29, 30, 32, 35, 36, 38, 41, 42, 48]                   b'\x14\xdb\x01\x01\x00\x03\xc3\x01\x0f\x04\xb5(\x00\xe4\xf5\x95D\x03V\x1c \xa7\x1c \xb6\x10d1\xf3\x15\xb0\x12\x03\xde@\x7ff\x07s\x8c\xf5\x1c \xd2P\x16q\x088 \x03\xc2\x08\x04\x18\xcd\x02\xd7\xc3Y@\x07\x03\x07\x02\x02\x03\x07\x02\x02\x0f\x03\x1a\x07\xc4\x08X\x05\x19\x00'
          2   21 [1, 2, 3, 4, 7, 11, 12, 14, 16, 17, 18, 19, 21, 23, 24, 26, 28, 29, 30, 32, 35, 36, 38, 41, 42, 48] b'\x14\xdb\x01\x01\x00\x03\xd5\x01\x0e\x05\x97x\x00?\xe2\xc9L\xa2[\x1c \x

## Block 4 — Build the UAP decoder map and instantiate data item decoders
This cell prepares the UAP reference table, dynamically discovers the decoder classes available for each ASTERIX category and item, creates one decoder instance per UAP entry, and stores the resulting decoder objects for later use.

In [ ]:

from asterix_decoder.data_tables.uap_tables import uap021_df, uap048_df
import importlib.util

def load_uap(path: Path, cat: int) -> pd.DataFrame:
    pass
    """Load and normalize UAP file into: cat, frn, item_id, item_name, length_str."""
    df = pd.read_csv(path) if path.suffix == ".csv" else pd.read_excel(path)

    # Normalize column names
    df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

    # Accept common variants
    rename_map = {
        "frn": "frn",
        "data_item": "item_id",
        "item_id": "item_id",
        "description": "item_name",
        "length_in_octets": "length_str",
    }
    df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

    required = {"frn", "item_id", "item_name", "length_str"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing UAP columns in {path}: {missing}")

    frn_str = df["frn"].astype(str).str.strip()
    valid = frn_str.str.fullmatch(r"\d+") & ~df["length_str"].astype(str).str.strip().isin(["-"])
    df = df[valid].copy()

    return pd.DataFrame(
        {
            "cat": cat,
            "frn": frn_str[valid].astype(int).values,
            "item_id": df["item_id"].astype(str).str.strip().values,
            "item_name": df["item_name"].fillna("").astype(str).str.strip().values,
            "length_str": df["length_str"].fillna("").astype(str).str.strip().values,
        }
    )


def discover_item_classes(cats: list[int], base_folder: Path = Path("asterix_decoder/data_items")) -> dict[tuple[int, str], type]:
    """Map (cat, item_id) -> class loaded dynamically from CATxxx folders."""
    class_map: dict[tuple[int, str], type] = {}

    for cat in cats:
        folder = base_folder / f"CAT{cat:03d}"   # CAT021, CAT048, ...
        if not folder.exists():
            continue

        for file in folder.glob("*.py"):
            if file.name == "__init__.py":
                continue

            module_name = f"dyn_cat{cat:03d}_{file.stem}"
            spec = importlib.util.spec_from_file_location(module_name, file)
            if spec is None or spec.loader is None:
                continue

            module = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(module)

            for _, cls in inspect.getmembers(module, inspect.isclass):
                if cls.__module__ == module_name and callable(getattr(cls, "get_item_id", None)):
                    item_id = str(cls.get_item_id()).strip()
                    class_map[(cat, item_id)] = cls

    return class_map


def build_instance(row: pd.Series, class_map: dict[tuple[int, str], type]):
    """Create one decoder instance for one UAP r## Block 7 — Export the decoded data to CSV\n\nThis cell creates the output folder if needed, exports the filtered DataFrame to a CSV file using the configured separator and decimal format, and prints a short summary of the exported result.ow."""
    cat = int(row["cat"])
    item_id = str(row["item_id"]).strip()
    item_name = str(row["item_name"])
    length_str = str(row["length_str"])

    cls = class_map.get((cat, item_id))
    if cls is None:
        return ItemXXX(item_name=item_name, length_str=length_str, item_id=item_id)

    try:
        return cls(item_name=item_name, length_str=length_str)
    except TypeError:
        return cls(item_name, length_str)


uap_df = pd.concat(
    [uap021_df, uap048_df],
    ignore_index=True,
)

CLASS_MAP = discover_item_classes([21, 48])
uap_df["instance"] = uap_df.apply(lambda r: build_instance(r, CLASS_MAP), axis=1)

print(uap_df[["cat", "frn", "item_id", "instance"]].head(10).to_string(index=False))

NameError: name 'importlib' is not defined

## Block 5 — Decode each message into structured fields
This cell defines the message decoding routine, decodes each message field by field using the FRN-to-decoder map, computes derived values such as target latitude/longitude and corrected Mode C altitude when possible, and stores the decoded output in the DataFrame.

In [ ]:
from asterix_decoder.helpers.compute_target_lat_lon import compute_target_lat_lon
def decode_message(cat: int, frns: list[int], data_fields: bytes, frn_map: dict) -> dict[str, any]:
    """Versión sin pandas para máxima velocidad."""
    final_data = {}
    cursor = 0
    for frn in frns:
        instance = frn_map.get((cat, frn))
        delta_cursor, instance_data = instance.decode(data_fields[cursor:])

        final_data.update(instance_data)
        cursor += delta_cursor
    
    RHO = final_data.get("RHO", None)
    THETA = final_data.get("THETA", None)
    FL = final_data.get("FL", None)
    if RHO is not None:
        lat, lon = compute_target_lat_lon(RHO, THETA, FL)
        final_data["LAT"] = lat
        final_data["LON"] = lon
    
    if FL is not None:
        BP = final_data.get("BP", None)
        if BP is not None and FL < 60:
            final_data["MODE_C_CORRECTED"] = round(final_data["FL"]*100+(BP-1013.2)*30,2)
    else:
        final_data["H(m)"] = 0
        final_data["H(ft)"] = 0
    
    return final_data

# Precompila el mapa (cat, frn) → instance
frn_map = dict(zip(uap_df[["cat", "frn"]].apply(tuple, axis=1), uap_df["instance"]))

# Aplica sin pandas overhead
messages_df["data"] = [
    decode_message(r.cat, r.frns, r.data_fields, frn_map)
    for r in messages_df.itertuples(index=False)
]

## Block 6 — Build the final table and apply filters
This cell selects the output columns based on the categories present, converts the decoded dictionaries into a tabular DataFrame, adds the CAT label, filters messages to the Barcelona TMA bounding box, removes unwanted GBS records, and leaves an optional filter for specific target types commented out.

In [ ]:
from pathlib import Path
import pandas as pd
from asterix_decoder.data_tables.csv_table import CSV_CAT021_COLUMNS, CSV_CAT048_COLUMNS

cats_presentes = set(pd.to_numeric(messages_df["cat"], errors="coerce").dropna().astype(int).unique())

if cats_presentes == {48}:
    target_cols = CSV_CAT048_COLUMNS.copy()
elif cats_presentes == {21}:
    target_cols = CSV_CAT021_COLUMNS.copy()
    target_cols.append("GBS")
else:
    shared = set(CSV_CAT021_COLUMNS).intersection(CSV_CAT048_COLUMNS)
    target_cols = [c for c in CSV_CAT048_COLUMNS if c in shared]

target_cols = [c for c in target_cols if c != "CAT"]
target_cols = ["CAT"] + target_cols

# --- Build rows from decoded dicts ---
rows = []
for r in messages_df.itertuples(index=False):
    data_dict = r.data if isinstance(r.data, dict) else {}
    row_out = {col: data_dict.get(col, None) for col in target_cols[1:]}
    rows.append(row_out)

final_df = pd.DataFrame(rows, columns=target_cols[1:])

# --- CAT column ---
cat_series = pd.to_numeric(messages_df["cat"], errors="coerce").astype("Int64")
final_df.insert(0, "CAT", cat_series.map(lambda x: f"CAT0{x}" if pd.notna(x) else pd.NA))

# ==============================================================================
# FILTERS
# ==============================================================================

# ── Filter 1 · Bounding box Barcelona TMA ─────────────────────────────────────

LAT_MIN, LAT_MAX = 40.9, 41.7
LON_MIN, LON_MAX = 1.5, 2.6

lat_col = next((c for c in final_df.columns if "LAT" in c.upper()), None)
lon_col = next((c for c in final_df.columns if "LON" in c.upper()), None)

lat = pd.to_numeric(final_df[lat_col], errors="coerce")
lon = pd.to_numeric(final_df[lon_col], errors="coerce")

in_bbox =  lat.isna() | (lat.between(LAT_MIN, LAT_MAX) & lon.between(LON_MIN, LON_MAX))
final_df = final_df[in_bbox].reset_index(drop=True)

# ── Filter 2 · Discard GBS at 0 elimnate ───────────────────────────────────-
final_df = final_df[~(final_df["GBS"] == 1)].reset_index(drop=True)
final_df = final_df.drop(columns=["GBS"], errors="ignore")
 

# ── Filter 3 · Discard unwanted TYP_020 values (Optional) ────────────────────────────────
# TYP_020_DISCARD = {
#     "No detection",
#     "PSR",
#     "SSR",
#     "SSR + PSR",
# }

# keep_typ = ~final_df["TYP_020"].isin(TYP_020_DISCARD)
# final_df = final_df[keep_typ].reset_index(drop=True)

# Reemplazar strings vacíos por nulos, pero sin convertir todo a texto
# final_df = final_df.replace(r"^\s*$", pd.NA, regex=True)


## Block 7 — Export the decoded data to CSV
This cell creates the output folder if needed, exports the filtered DataFrame to a CSV file using the configured separator and decimal format, and prints a short summary of the exported result.

In [ ]:
# ==============================================================================
# EXPORT CSV
# ==============================================================================

output_dir = Path("output")
output_dir.mkdir(exist_ok=True)

csv_path = output_dir / "decoded_messages.csv"

final_df.to_csv(csv_path, index=False, sep=";", decimal=",", na_rep="N/A")

print(f"CATS presentes   : {sorted(cats_presentes)}")
print(f"Columnas finales : {len(final_df.columns)}")
print(f"Shape final_df   : {final_df.shape}")
print(f"✓ CSV  → {csv_path}")